<a href="https://colab.research.google.com/github/ANKIT-KANDULNA/ANKIT-KANDULNA/blob/main/Experiment-3/DL_Lab_exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Imports

In [9]:
!pip install datasets -q

In [10]:
from datasets import load_dataset

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(len(train_data), len(test_data))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

25000 25000


Tokenizer + Vocabulary

In [11]:
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

counter = Counter()

for sample in train_data:
    counter.update(tokenize(sample['text']))

vocab = {"<pad>":0, "<unk>":1}

for word, freq in counter.items():
    if freq > 2:   # filter rare words
        vocab[word] = len(vocab)

print("Vocab size:", len(vocab))

Vocab size: 37895


Encode

In [12]:
def encode(text):
    tokens = tokenize(text)
    return [vocab.get(t, vocab["<unk>"]) for t in tokens]

Dataset Class

In [13]:
import torch
from torch.utils.data import Dataset

class IMDBDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = encode(self.data[idx]['text'])
        label = self.data[idx]['label']
        return torch.tensor(text), torch.tensor(label, dtype=torch.float32)

Collate Function (Padding)

In [14]:
from torch.nn.utils.rnn import pad_sequence

def collate(batch):
    texts, labels = zip(*batch)

    texts = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)

    return texts, labels

DataLoaders

In [15]:
from torch.utils.data import DataLoader

train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate)

# 1D CNN Model

In [16]:
import torch.nn as nn

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.conv1 = nn.Conv1d(embed_dim, 100, 3)
        self.conv2 = nn.Conv1d(embed_dim, 100, 4)
        self.conv3 = nn.Conv1d(embed_dim, 100, 5)

        self.fc = nn.Linear(300, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.embedding(x)      # (B,L,E)
        x = x.permute(0,2,1)       # (B,E,L)

        c1 = torch.relu(self.conv1(x))
        c2 = torch.relu(self.conv2(x))
        c3 = torch.relu(self.conv3(x))

        p1 = torch.max(c1, dim=2)[0]
        p2 = torch.max(c2, dim=2)[0]
        p3 = torch.max(c3, dim=2)[0]

        x = torch.cat([p1,p2,p3], dim=1)

        x = self.dropout(x)
        return self.fc(x).squeeze(1)

Train Setup

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TextCNN(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Training Function

In [18]:
def train():
    model.train()
    total = 0

    for x,y in train_loader:
        x,y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total += loss.item()

    return total/len(train_loader)

# Evaluation

In [19]:
def evaluate():
    model.eval()
    correct,total = 0,0

    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)

            out = model(x)
            pred = torch.round(torch.sigmoid(out))

            correct += (pred==y).sum().item()
            total += y.size(0)

    return correct/total

# Running

In [20]:
for epoch in range(5):
    loss = train()
    acc = evaluate()

    print(f"Epoch {epoch+1} Loss:{loss:.4f} Acc:{acc:.4f}")

Epoch 1 Loss:0.6170 Acc:0.7818
Epoch 2 Loss:0.4537 Acc:0.8536
Epoch 3 Loss:0.3732 Acc:0.8772
Epoch 4 Loss:0.3163 Acc:0.8829
Epoch 5 Loss:0.2637 Acc:0.8881
